# Notebook Docente 04 — Óptico (Sentinel-2) vs. Radar (Sentinel-1) (Bloque 6)
## Teledetección — Maestría en Ingeniería, Universidad del Magdalena

> ⚠️ **Este notebook es una DEMO DEL DOCENTE** — durante la Sesión 1 el docente lo proyecta
> en pantalla mientras explica la teoría. Los estudiantes NO lo corren en clase.
>
> **Para estudiantes:** puedes abrirlo después de la sesión para repasar. Todo está comentado en español.
> No necesitas ejecutar nada — puedes leer las celdas y los resultados de la demostración.

---

**Sesión 1 · Viernes 17 de julio de 2026**

---

**Este notebook es para que TÚ (el docente) lo proyectes en pantalla.** Versión en
Python/Colab del script `gee/04_optico_vs_sar.js`.

### Qué vas a mostrar
Buscamos a propósito un mes de temporada de lluvias en el norte del Magdalena, donde
Sentinel-2 (sensor pasivo, óptico) sale cubierto de nubes, mientras Sentinel-1
(sensor activo, radar SAR) entrega la misma zona completamente visible el mismo mes.
Es la demostración más directa de la diferencia entre sensor pasivo y sensor activo
(Bloque 5) y de por qué el Caribe colombiano necesita SAR (Bloque 6).

In [1]:
!pip install geemap --quiet
print("✓ Instalación completada")

✓ Instalación completada


In [2]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='teledeteccion-miguepoloc')  # reemplaza con tu ID de proyecto GEE

print("✓ Google Earth Engine inicializado correctamente")

✓ Google Earth Engine inicializado correctamente


## Paso 1 — Sentinel-2 en temporada de lluvias (sin filtrar nubes, a propósito)

In [3]:
norte_magdalena = ee.Geometry.Rectangle([-74.5, 10.2, -73.2, 11.2])

coleccion_s2_lluvias = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(norte_magdalena)
    .filterDate('2024-10-01', '2024-10-31')
)

# .mosaic() en vez de .first(): un solo tile Sentinel-2 (~110x110 km) no
# alcanza a cubrir todo norte_magdalena (~130x110 km) -- con .first() solo
# se veía el pedacito de un tile. .mosaic() combina TODAS las escenas de
# octubre-2024 que caen en la zona (varias franjas/tiles) en un mosaico que
# sí cubre el rectángulo completo, sin enmascarar nubes (a propósito).
s2_lluvias = coleccion_s2_lluvias.mosaic().clip(norte_magdalena)

n_imagenes_lluvias = coleccion_s2_lluvias.size().getInfo()
pct_nubes_promedio = coleccion_s2_lluvias.aggregate_mean('CLOUDY_PIXEL_PERCENTAGE').getInfo()

print(f'Imágenes Sentinel-2 combinadas en el mosaico de octubre-2024: {n_imagenes_lluvias}')
print(f'% de nubes promedio entre esas imágenes: {pct_nubes_promedio:.1f}%')

Imágenes Sentinel-2 combinadas en el mosaico de octubre-2024: 31
% de nubes promedio entre esas imágenes: 55.5%


## Paso 2 — Sentinel-2 en temporada seca (para contraste, no todo es malo)

In [4]:
s2_seca = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(norte_magdalena)
    .filterDate('2024-02-01', '2024-02-28')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
    .median()
    .clip(norte_magdalena)
)
print('✓ Imagen de temporada seca lista')

✓ Imagen de temporada seca lista


## Paso 3 — Sentinel-1 SAR, misma zona y mismo mes lluvioso

In [5]:
s1_lluvias = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(norte_magdalena)
    .filterDate('2024-10-01', '2024-10-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .median()
    .clip(norte_magdalena)
)
print('✓ Imagen Sentinel-1 SAR lista')

✓ Imagen Sentinel-1 SAR lista


## Paso 4 — Mapa comparativo (activa/desactiva capas en el panel de capas)

In [6]:
m = geemap.Map()
m.centerObject(norte_magdalena, 10)

m.addLayer(s2_lluvias, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4},
           '1 — Sentinel-2 óptico (LLUVIAS, oct-2024)')
m.addLayer(s1_lluvias, {'bands': ['VV'], 'min': -20, 'max': 0},
           '2 — Sentinel-1 SAR (misma zona y mes)', False)
m.addLayer(s2_seca, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000, 'gamma': 1.4},
           '3 — Sentinel-2 óptico (SECA, feb-2024, contraste)', False)

m.addLayerControl()
m

Map(center=[10.700397756578358, -73.85], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

## Guion para la clase

1. Muestra la capa 1 (Sentinel-2 lluvias) — nubes cubren gran parte de la escena.
2. Apaga la 1, enciende la 2 (Sentinel-1 SAR) — la superficie es completamente visible.
3. Enciende la 3 (Sentinel-2 seca) — así sí funciona el óptico, para no dar la idea de
   que el sensor pasivo "nunca sirve".

**Pregunta para la clase:** ¿por qué Sentinel-1 "ve" a través de las nubes y Sentinel-2 no?
*Respuesta (Bloque 2 + Bloque 6):* la onda de radar (5.6 cm) es miles de veces más grande
que las gotas de nube (5-100 μm) y las atraviesa; la luz visible es del tamaño de esas
gotas y rebota en ellas.

**Dato regional:** el Caribe colombiano tiene 60-80% de días nublados en temporada de
lluvias — por eso el Artículo 3 de investigación del docente fusiona Sentinel-1 y
Sentinel-2 en vez de usar solo uno.